# Upload scraped-jobs JSON to MongoDB

When the backend API isn't running during a scrape, the Chrome extension downloads the jobs as a JSON file (e.g. `glassdoor_jobs_2026-07-11.json`) instead of POSTing them to `/jobs`.

This notebook uploads such a file into MongoDB using the **same `insert_jobs()` function the API uses**, so everything stays consistent:

- Duplicates are skipped automatically (upsert by `job_id`)
- `posted` strings like `"24h"` / `"30d+"` are converted to a real `posted_at` datetime
- Jobs are stored with `llm_processed: False` and `llm_status: "pending"`

After running this, open the Streamlit dashboard → **Not Processed** page → click **▶ Process N jobs (Free Tier)** to run the AI scoring, then apply via the **LLM Processed** page as usual.

**Prerequisites:** MongoDB must be running (`mongod`), and this notebook must use the same Python environment as the backend (`backend/env`).

In [1]:
# ── Config: point this at the JSON file you want to upload ──
JSON_FILE = r"C:\Users\tarun\Downloads\glassdoor_jobs_2026-07-11.json"

# Works for both glassdoor_jobs_*.json and linkedin_jobs_*.json files.

In [2]:
# ── Load the file and show what's inside ──
import json
from pathlib import Path

path = Path(JSON_FILE)
assert path.exists(), f"File not found: {path}"

with open(path, encoding="utf-8") as f:
    payload = json.load(f)

jobs = payload["jobs"]
print(f"Source       : {payload.get('source')}")
print(f"Scraped date : {payload.get('scraped_date')}")
print(f"Total jobs   : {payload.get('total')} (jobs array: {len(jobs)})")
print(f"Jobs with JD : {sum(1 for j in jobs if j.get('jd'))}")
print()
print("First 5 jobs:")
for j in jobs[:5]:
    print(f"  - {j.get('title')} @ {j.get('company')} ({j.get('location')})")

Source       : glassdoor
Scraped date : 2026-07-11
Total jobs   : 198 (jobs array: 198)
Jobs with JD : 198

First 5 jobs:
  - Analyst – Global Performance Management & Analytics @ Kenvue (India)
  - Associate AI/ML Engineer @ UnitedHealth Group (Hyderābād)
  - AI/ML Engineer @ UnitedHealth Group (India)
  - Career Expert - Fashion Design @ Academy of Design and Innovation (ADI) (India)
  - Architect - python, nodejs/java, react js, AI/ML @ UnitedHealth Group (India)


In [3]:
# ── Upload to MongoDB via the backend's own insert_jobs() ──
import sys

BACKEND_DIR = r"C:\Users\tarun\OneDrive\Desktop\AI-Based-Job-Filter\backend"
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

from db import insert_jobs

inserted_ids = insert_jobs(jobs)

print(f"✅ Inserted : {len(inserted_ids)} new jobs")
print(f"⏭️ Skipped  : {len(jobs) - len(inserted_ids)} duplicates (already in DB)")

✅ Inserted : 180 new jobs
⏭️ Skipped  : 18 duplicates (already in DB)


In [4]:
# ── Verify: how many jobs from this file's date are now pending? ──
from datetime import datetime
from db import get_collection

col = get_collection()
date_str = payload.get("scraped_date")
start = datetime.fromisoformat(f"{date_str}T00:00:00+00:00")
end   = datetime.fromisoformat(f"{date_str}T23:59:59+00:00")

total_for_date = col.count_documents({"scraped_at": {"$gte": start, "$lte": end}})
pending        = col.count_documents({
    "scraped_at": {"$gte": start, "$lte": end},
    "llm_status": "pending",
})

print(f"Jobs in DB scraped on {date_str} : {total_for_date}")
print(f"Pending LLM processing           : {pending}")
print()
print("Next: run `streamlit run app.py`, pick this date in the sidebar,")
print("open the 'Not Processed' page and click '▶ Process N jobs (Free Tier)'.")

Jobs in DB scraped on 2026-07-11 : 180
Pending LLM processing           : 180

Next: run `streamlit run app.py`, pick this date in the sidebar,
open the 'Not Processed' page and click '▶ Process N jobs (Free Tier)'.
